# Step6b — masks, coverage, and final analytical ROIs for FC

**Main QC overview:** the notebook displays ROI-level inclusion and a PASS/FAIL matrix for all configured `mask_set` comparisons and the two independent REST branches.

- **Analysis-level PASS** = `selected_rois_by_mask_set.csv: include=True`: the technical representation of an ROI meets the cohort-level selection criteria. This **does not** imply that every individual stem passes.
- **Stem-level PASS** = `homotopic_stem_pass` or `global_stem_pass` for an individual recording; the corresponding stem usability tables are evaluated separately.
- Example mask sets: `stroke_longitudinal` can use `healthy` for global analysis, whereas `prestroke_stability` and `daypt_anesthesia` may select separate left/right sides according to their YAML configuration. These names and choices are examples, not universal requirements.
- `global_healthy_*.npy` remains a **healthy-side-only** product. `global_analysis_*.npy` reflects the selected analysis hemisphere (`both` combines left and right), without renaming the legacy masks.

The remaining sections provide visual QC for a selected mouse: canonical and reference atlas projections, brain-edge erosion, hemorrhage/shadow exclusions, and geometric consistency. Selection values such as mouse ID, ROI and mask set are examples and must correspond to the configured data.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# EDIT THE PROJECT-SPECIFIC SETTINGS BELOW
# ============================================================

# Expected project layout: project/notebook/ and project/src/.
# Start this notebook with the Jupyter working directory set to project/notebook/.
NOTEBOOK_DIR = Path.cwd().resolve()
SRC_DIR = (NOTEBOOK_DIR.parent / "src").resolve()

required_files = ("common.py", "step6_atlas_transformation.py", "step6b_fc_geometry_coverage.py")
missing_files = [name for name in required_files if not (SRC_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing source files: {', '.join(missing_files)}. Expected src/ directory: {SRC_DIR}. "
        "Run this notebook from the project's notebook/ directory."
    )

# Example configuration path: replace with your own project YAML.
CONFIG_PATH = Path(r"config.yaml")

# Example: choose the REST branch used for QC. The available branches are independent.
REST_VARIANT = "with_rest"  # QC branch: without_rest | with_rest
RUN_BOTH_VARIANTS = True    # when RUN_BATCH=True, process both branches; False processes one

# Example QC selections; set these to identifiers present in your analysis
MASK_SET = "stroke_longitudinal"
QC_MOUSE = "m18"
QC_ROI = "MOp_right"  # if MO refinement is enabled; otherwise use e.g. "MO_left"

# None = choose the first available selected stem for this mouse/mask_set
QC_STEM = None

# False loads previously saved tables without rebuilding masks.
# True runs full Step6b and writes outputs with overwrite=True.
RUN_BATCH = True  # WARNING: the current setting enables a full batch recalculation

# ============================================================

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import step6b_fc_geometry_coverage as s6b

print("Step6b:", Path(s6b.__file__).resolve())
print("Config:", CONFIG_PATH)
print("Mask set:", MASK_SET)
print("REST QC variant:", REST_VARIANT)
print("QC mouse:", QC_MOUSE)
print("QC ROI:", QC_ROI)


## 1. Dry-run selection: what will enter each mask set

In [ ]:
# Compare BOTH independent selection lists before constructing masks.
# Show concise summaries by default; detailed selections remain available via `dry_both`.
dry_both = s6b.run_step6b(CONFIG_PATH, dry_run=True, rest_variant="both")
for branch in ("without_rest", "with_rest"):
    dry = dry_both[branch]
    print(f"\n=== {branch}: selection overview ===")
    display(dry["cohort_overview"])
    print("Missing/ambiguous/no REST slots:")
    skipped = dry["skipped_mask_sets"]
    display(skipped[[c for c in ["mask_set", "mouse_id", "n_ok", "n_unresolved", "unresolved_status_json"] if c in skipped.columns]])
    print("Scientific exclusion rule counts:")
    display(dry["exclusion_rules"])
# To inspect every slot: display(dry_both["with_rest"]["selection"])


## 2. Run or load Step6b

Before running Step6b, verify that the anatomical left/right convention in the Step6 outputs agrees with the configured atlas orientation. The setting is read from `step6.atlas.ml_sign` in the project YAML.

**Experiment-specific example:** an earlier canonical-to-reference diagnostic suggested `step6.atlas.ml_sign = -1.0` for a particular dataset. This is **not** a recommended value for every project. If the correct atlas-orientation setting is changed, regenerate the affected Step6 atlas outputs before using them in Step6b.

**Important:** `RUN_BATCH = True` calls `run_step6b(..., overwrite=True)` and can overwrite existing Step6b outputs. Set `RUN_BATCH = False` to load saved tables without rebuilding geometry. `RUN_BOTH_VARIANTS` controls whether both independent REST branches are processed when a batch run is enabled.


In [ ]:
if RUN_BATCH:
    if RUN_BOTH_VARIANTS:
        all_results = s6b.run_step6b(CONFIG_PATH, dry_run=False, overwrite=True, rest_variant="both")
        result = all_results[REST_VARIANT]
        print("Both independent branches completed.")
    else:
        result = s6b.run_step6b(CONFIG_PATH, dry_run=False, overwrite=True, rest_variant=REST_VARIANT)
        print(f"Branch {REST_VARIANT} completed.")
else:
    result = s6b.load_step6b_tables(CONFIG_PATH, rest_variant=REST_VARIANT)
    print("Loaded saved branch", REST_VARIANT)

_, config, paths = s6b.build_step6_index(CONFIG_PATH, include_bad=True, require_response_roi=False)
s6b.ensure_step6b_defaults(config)
config["step6b_fc_geometry"]["active_rest_variant"] = REST_VARIANT
out_paths = s6b.build_step6b_paths(config, paths)
print("Step6b root:", out_paths["root"])


## Main result — which analytical ROIs pass cohort selection?

The tables below are **loaded from saved Step6b outputs**. These QC cells do not rebuild masks or modify thresholds. If a REST branch has not been generated, the notebook reports that its results are missing.

First, a compact table lists the included regions (one row per `mask_set × spatial analysis × ROI`). Next, a full PASS/FAIL matrix lists all candidate regions. For the `daypt_anesthesia` comparison, Awake and Iso are considered separately, rather than using the original `Control / Awake_PT / Iso_PT / UNASSIGNED` experimental-group labels as comparison classes.

**Cohort-level PASS does not guarantee stem-level PASS.** The notebook separately reports ROIs available for the selected mouse.


In [ ]:
from IPython.display import display

qc_branches = {}
for branch in ("without_rest", "with_rest"):
    # Works with old saved tables too; full Step6b rerun is NOT necessary just to display.
    tb = s6b.load_step6b_tables(CONFIG_PATH, rest_variant=branch)
    selected_all = tb.get("selected_rois", pd.DataFrame())
    groups_all = tb.get("group_summary", pd.DataFrame())
    if selected_all.empty:
        print(f"{branch}: selected_rois_by_mask_set.csv not found; the full Step6b run has not completed")
        continue
    summary = s6b.build_final_roi_pass_summary(selected_all, groups_all)
    qc_branches[branch] = {"tables": tb, "summary": summary}
    print(f"\n========== {branch} ==========")
    print("PASS — ROIs included by cohort-level selection for Step7:")
    accepted = summary.loc[summary["PASS"]].copy()
    display(accepted[["mask_set", "analysis", "roi_key", "group_technical_pass_details"]]
            .sort_values(["mask_set", "analysis", "roi_key"]).reset_index(drop=True))
    count = (summary.groupby(["mask_set", "analysis"], dropna=False)["PASS"]
             .agg(passed="sum", total="count").reset_index())
    count["rejected"] = count["total"] - count["passed"]
    print("Counts of included and excluded ROIs:")
    display(count)

    # One row per broad Allen region, one column per side / FC analysis.
    matrix = summary.copy()
    matrix["product"] = np.where(matrix["analysis"].eq("homotopic"),
                                  "homotopic_LR", "global_" + matrix["hemisphere"].astype(str))
    pivot = (matrix.pivot_table(index=["mask_set", "roi"], columns="product", values="PASS", aggfunc="first")
             .sort_index())
    # Keep not-applicable cells distinct from scientific FAIL.
    readable = pivot.apply(lambda col: col.map({True: "PASS", False: "FAIL"}))
    print("ROI MATRIX (— = spatial product not applicable to this analysis):")
    def _qc_style(value):
        return ("background-color: #d9eee2; color: #134b2c" if value == "PASS" else
                "background-color: #fae0e0; color: #842222" if value == "FAIL" else "")
    display(readable.fillna("—").style.applymap(_qc_style))
    print("Excluded ROIs and recorded reasons (QC):")
    rejected = summary.loc[~summary["PASS"]]
    display(rejected[["mask_set", "analysis", "roi_key", "min_group_technical_pass_fraction",
                      "required_fraction", "group_technical_pass_details", "reason"]]
            .reset_index(drop=True))

if not qc_branches:
    print("No tables from a completed run were found. Set RUN_BATCH=True and execute Step6b to generate them.")


### Mouse counts and per-stem eligibility for the selected mouse

- `n_mice_considered` counts unique mice **within a mask set**, not the entire Step2 cohort.
- `selection_complete` indicates whether the required temporal slots were found.
- `masks_built` indicates whether the required spatial masks were successfully constructed.
- `n_selected_stems` counts selected recording stems, **not** animals.

For an example Day-PT anesthesia comparison, the overview distinguishes **Awake** and **Iso**. The original experimental-group assignment does not replace that comparison. Stem-level PASS fields are reported separately from cohort-level statistics.


In [ ]:
for branch, record in qc_branches.items():
    tb = record["tables"]
    print(f"\n=== {branch} / mouse counts ===")
    # Old Step6b tables: reconstruct overview without recompiling geometry.
    overview = s6b.build_cohort_overview(tb.get("selection", pd.DataFrame()), config,
                                          tb.get("mask_set_summary", pd.DataFrame()))
    display(overview.reset_index(drop=True))
    print(f"Mouse={QC_MOUSE} / mask_set={MASK_SET}: individually passing ROI among COHORT selected")
    eligible = tb.get("eligibility", pd.DataFrame())
    selected_mask_set = record["summary"]
    accepted = selected_mask_set[(selected_mask_set["mask_set"].eq(MASK_SET)) & selected_mask_set["PASS"]]
    if eligible.empty or accepted.empty:
        print("No eligibility or cohort PASS data for this selection")
        continue
    e = eligible[(eligible["mouse_id"].astype(str) == str(QC_MOUSE)) &
                 (eligible["mask_set"].astype(str) == str(MASK_SET))].copy()
    if e.empty:
        print("No mask set was built for this mouse")
        continue
    e = e.merge(accepted[["mask_set", "analysis", "roi_key"]],
                on=["mask_set", "analysis", "roi_key"], how="inner")
    columns = ["analysis", "roi_key", "technical_pass", "relative_to_cohort",
               "n_stems_pass", "n_selected_stems", "any_selected_stem_pass"]
    display(e[[col for col in columns if col in e.columns]].sort_values(["analysis", "roi_key"]).reset_index(drop=True))


### Diagnose a FAIL for a selected mouse and ROI (example: `m14 / RSP`)

This is **fixed technical coverage before stem-specific shadow/hemorrhage masking**, which contributes to cohort-level ROI inclusion. In particular, `min_relative_fraction` is applied to coverage **relative to the per-mask-set cohort reference statistic** (median by default, if configured accordingly).

The example mouse/ROI variables below can be changed for reviewing saved tables without a full Step6b rerun.


In [ ]:
AUDIT_MOUSE = "m14"
AUDIT_ROI = "RSP"

for branch, rec in qc_branches.items():
    tb = rec["tables"]
    pieces = []
    for kind, name, key_col, px_col, rel_col, pass_col in [
        ("homotopic", "homotopic_technical_coverage", "pair_key",
         "canonical_homotopic_technical_pair_px", "homotopic_relative_to_cohort", "homotopic_technical_pass"),
        ("global", "global_technical_coverage", "global_roi_key",
         "canonical_technical_valid_px", "global_relative_to_cohort", "global_technical_pass"),
    ]:
        df = tb.get(name, pd.DataFrame())
        if df.empty: continue
        part = df[df["mouse_id"].astype(str).eq(AUDIT_MOUSE) &
                  df[key_col].astype(str).str.startswith(AUDIT_ROI + "_")].copy()
        if part.empty: continue
        part["analysis"] = kind
        part["roi_key"] = part[key_col]
        part["technical_pixels_or_pairs"] = pd.to_numeric(part[px_col], errors="coerce")
        part["relative_to_median"] = pd.to_numeric(part[rel_col], errors="coerce")
        part["technical_PASS"] = part[pass_col].astype(bool)
        part["required_relative_min"] = part["mask_set"].map(lambda ms:
            float(s6b.thresholds_for_mask_set(config, str(ms))["cohort_relative"]["min_relative_fraction"]))
        pieces.append(part[["mask_set", "analysis", "roi_key", "technical_pixels_or_pairs",
                            "relative_to_median", "required_relative_min", "technical_PASS"]])
    print(f"\n{branch}: mouse={AUDIT_MOUSE}, ROI={AUDIT_ROI}")
    if pieces:
        detail = pd.concat(pieces, ignore_index=True)
        display(detail.sort_values(["mask_set", "analysis", "roi_key"]).style.format(
            {"relative_to_median": "{:.1%}", "required_relative_min": "{:.0%}"}))
    else:
        print("No technical coverage tables for this mouse/ROI")


## 3. Left/right projection consistency

In [ ]:
lr = result.get("left_right_consistency", pd.DataFrame()).copy()

if lr.empty:
    print("No left/right consistency table yet. Run Step6b v4.12.")
else:
    cols = [
        "mask_set", "mouse_id", "group_id",
        "atlas_ml_sign",
        "atlas_left_ml_relation_to_bregma",
        "image_ml_to_image_sign",
        "image_ml_positive_label",
        "canonical_left_to_ref_left",
        "canonical_left_to_ref_right",
        "canonical_right_to_ref_left",
        "canonical_right_to_ref_right",
        "expected_minus_opposite",
        "side_mapping_status",
    ]
    display(
        lr[[c for c in cols if c in lr.columns]]
        .sort_values(["mask_set", "mouse_id"])
        .reset_index(drop=True)
    )

    bad = lr[~lr["side_mapping_status"].astype(str).eq("ok")]
    if bad.empty:
        print("OK: all processed mouse × mask_set mappings are consistent.")
    else:
        print("ATTENTION: non-OK left/right mappings:")
        display(bad)


## 4. Canonical Allen ROI geometry

In [ ]:
# Build canonical analytical atlas ONCE before all canonical QC sections.
atlas_data = s6b.load_allen_atlas(config)
parent_query = s6b.cfg_get(
    config,
    "step6b_fc_geometry",
    "broad_atlas",
    "parent_acronym",
    default="Isocortex",
)
canonical = s6b.build_canonical_side_rois(
    atlas_data,
    config,
    paths,
    parent_query,
)

canonical_rois = result.get("canonical_rois", pd.DataFrame()).copy()

show_cols = [
    "broad_family_acronym", "analysis_roi_acronym", "hemisphere", "roi_key",
    "canonical_raw_full_area_px",
    "canonical_internal_zero_hole_filled_px",
    "canonical_internal_nonzero_hole_reassigned_px",
    "canonical_reassigned_away_px",
    "canonical_protected_hole_px",
    "full_allen_area_px",
    "buffered_full_allen_area_px",
    "canonical_boundary_excluded_px",
    "canonical_boundary_retained_fraction",
    "canonical_left_columns",
    "canonical_right_columns",
]
display(
    canonical_rois[[c for c in show_cols if c in canonical_rois.columns]]
    .sort_values(["broad_family_acronym", "hemisphere"])
    .reset_index(drop=True)
)


### Selective analytical ROI refinement audit

`broad_family_*` identifies an original broad Allen family (for example, `MO`).  
`analysis_roi_*` identifies the analytical ROI used downstream (for example, `MOp` and `MOs`).

If `step6b_fc_geometry.broad_atlas.roi_refinements` is empty, these levels coincide. **Example:** configuring `MO -> [MOp, MOs]` selectively refines `MO`; unlisted broad families remain unchanged.


In [ ]:
refinements = result.get("roi_refinements", pd.DataFrame()).copy()

print("Declared selective refinements:")
if refinements.empty:
    print("No selective ROI refinements configured.")
else:
    display(refinements.reset_index(drop=True))

roi_defs = result.get("analysis_roi_definitions", result.get("direct_children", pd.DataFrame())).copy()
definition_cols = [
    "broad_family_acronym",
    "analysis_roi_acronym",
    "analysis_roi_name",
    "roi_refinement_applied",
    "roi_definition_source",
    "analysis_structure_id",
]
definition_cols = [c for c in definition_cols if c in roi_defs.columns]

print("\nFinal analytical ROI definitions:")
display(
    roi_defs[definition_cols]
    .sort_values(["broad_family_acronym", "analysis_roi_acronym"])
    .reset_index(drop=True)
)


### Canonical internal-hole repair QC

Before the boundary buffer is applied, Step6b v4.12 fills **enclosed zero-label holes** inside each analytical side ROI.

Important:
- only pixels where the analytical atlas label is `0` may be filled;
- a pixel belonging to another Allen ROI is never overwritten;
- the repaired mask is then used for `full`, `buffered`, coverage, homotopic pairs and cohort frequency maps.

The panel below is useful for checking the known tiny hole in `MOs`.


In [ ]:
# Known example: MOs_right had one enclosed non-zero pixel.
HOLE_QC_ROI = "MOs_right" if "MOs_right" in canonical["roi_masks_full"] else QC_ROI

raw_roi = canonical["roi_masks_raw"][HOLE_QC_ROI].astype(bool)
repaired_roi = canonical["roi_masks_full"][HOLE_QC_ROI].astype(bool)
buffered_roi = canonical["roi_masks_buffered"][HOLE_QC_ROI].astype(bool)

gained = repaired_roi & (~raw_roi)
lost = raw_roi & (~repaired_roi)
zero_filled = gained & (canonical["broad_full_raw"] == 0)
nonzero_reassigned = gained & (canonical["broad_full_raw"] != 0)

row = canonical["roi_table"][
    canonical["roi_table"]["roi_key"].astype(str).eq(HOLE_QC_ROI)
].copy()

display(
    row[
        [
            "roi_key",
            "canonical_raw_full_area_px",
            "canonical_enclosed_hole_candidate_px",
            "canonical_internal_zero_hole_filled_px",
            "canonical_internal_nonzero_hole_reassigned_px",
            "canonical_reassigned_away_px",
            "canonical_protected_hole_px",
            "canonical_hole_claim_conflict_px",
            "canonical_nonzero_hole_original_labels_json",
            "canonical_full_area_px",
            "canonical_buffered_area_px",
            "canonical_boundary_excluded_px",
            "canonical_boundary_retained_fraction",
        ]
    ].reset_index(drop=True)
)

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

axes[0].imshow(raw_roi, cmap="gray", interpolation="nearest")
axes[0].set_title(f"{HOLE_QC_ROI}\nRAW")

axes[1].imshow(repaired_roi, cmap="gray", interpolation="nearest")
axes[1].imshow(np.where(zero_filled, 1, np.nan), cmap="Reds", alpha=0.95, interpolation="nearest")
axes[1].set_title(f"ZERO holes filled\n{int(zero_filled.sum())} px")

axes[2].imshow(repaired_roi, cmap="gray", interpolation="nearest")
axes[2].imshow(np.where(nonzero_reassigned, 1, np.nan), cmap="Purples", alpha=0.95, interpolation="nearest")
axes[2].set_title(
    "NON-ZERO islands reassigned\n"
    f"{int(nonzero_reassigned.sum())} px"
)

axes[3].imshow(buffered_roi, cmap="gray", interpolation="nearest")
axes[3].set_title("BUFFERED after repair")

axes[4].imshow(
    np.ma.masked_where(canonical["full_side_labels"] == 0, canonical["full_side_labels"]),
    cmap="gray",
    alpha=0.25,
    interpolation="nearest",
)
axes[4].imshow(np.where(repaired_roi, 1, np.nan), cmap="viridis", alpha=0.75, interpolation="nearest")
axes[4].set_title("Repaired ROI on canonical atlas")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Canonical repair log:")
repair_log = canonical.get("hole_repair_table", pd.DataFrame()).copy()
if repair_log.empty:
    print("No canonical hole repairs.")
else:
    display(repair_log.reset_index(drop=True))


## 5. Cohort frequency map in canonical Allen space

A fixed bilateral canonical mask is accumulated **once per included mouse** and divided by the number of mice in the selected cohort.

Each pixel represents the fraction of included mice with that fixed technical representation, for example:
- `1.0`: present in all mice;
- `0.8`: present in 80% of mice;
- `0.4`: present in 40% of mice;
- `0.0`: present in none.

Stem-specific shadow and pathology are **not** included in this fixed-mask frequency map.


In [ ]:
cohort_dir = out_paths["cohort_canonical_root"] / s6b.safe_key(MASK_SET)
freq_path = cohort_dir / "fixed_technical_frequency_canonical.npy"
count_path = cohort_dir / "fixed_technical_count_canonical.npy"
mice_path = cohort_dir / "included_mice.csv"

if not freq_path.exists():
    raise FileNotFoundError(
        f"Cohort frequency map not found:\n{freq_path}\nRun Step6b v4.12 first."
    )

freq = np.load(freq_path)
count = np.load(count_path)
included_mice = pd.read_csv(mice_path)

print("Included mice:", len(included_mice))
display(included_mice)

atlas_labels = canonical["buffered_broad"]

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(np.where(atlas_labels > 0, freq, np.nan), vmin=0, vmax=1, cmap="viridis")

for bid in [int(x) for x in np.unique(atlas_labels) if int(x) > 0]:
    m = atlas_labels == bid
    if m.any():
        ax.contour(m.astype(float), levels=[0.5], linewidths=0.4)

ax.set_title(f"{MASK_SET}: fraction of mice with canonical pixel valid")
ax.axis("off")
cb = fig.colorbar(im, ax=ax)
cb.set_label("fraction of mice")
plt.show()


## 6. Spatial representation of Allen ROIs across mice

In [ ]:
side = result.get("side_technical_coverage", pd.DataFrame()).copy()
side = side[side["mask_set"].astype(str).eq(MASK_SET)].copy()

if side.empty:
    raise ValueError(f"No side technical coverage rows for mask_set={MASK_SET}")

roi_summary = (
    side.groupby(["roi_key", "broad_acronym", "hemisphere"], dropna=False)
    .agg(
        n_mice=("mouse_id", "nunique"),
        median_coverage=("technical_coverage_fraction", "median"),
        q25=("technical_coverage_fraction", lambda x: pd.to_numeric(x, errors="coerce").quantile(0.25)),
        q75=("technical_coverage_fraction", lambda x: pd.to_numeric(x, errors="coerce").quantile(0.75)),
        min_coverage=("technical_coverage_fraction", "min"),
        max_coverage=("technical_coverage_fraction", "max"),
        cohort_median=("cohort_reference_technical_coverage", "first"),
        n_pixel_pass=("technical_pixel_pass", "sum"),
        n_technical_pass=("technical_pass", "sum"),
    )
    .reset_index()
    .sort_values("median_coverage", ascending=False)
)

display(roi_summary.reset_index(drop=True))

plot_df = roi_summary.sort_values("median_coverage").reset_index(drop=True)
y = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.28)))
x = plot_df["median_coverage"].to_numpy(float)
xerr = np.vstack([
    x - plot_df["q25"].to_numpy(float),
    plot_df["q75"].to_numpy(float) - x,
])
ax.errorbar(x, y, xerr=xerr, fmt="o", capsize=2)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["roi_key"])
ax.set_xlim(0, 1.02)
ax.set_xlabel("technical coverage fraction (median; IQR)")
ax.set_title(f"{MASK_SET}: ROI representation across mice")
ax.grid(axis="x", alpha=0.25)
plt.show()


### Selected ROI: individual mice vs cohort median

`relative_to_cohort` compares each mouse with the median coverage of the same `mask_set × ROI_side`.  
The high boundary is a flag by default; the lower boundary is the inclusion gate.


In [ ]:
roi = side[side["roi_key"].astype(str).eq(QC_ROI)].copy()
if roi.empty:
    raise ValueError(f"ROI {QC_ROI!r} is absent from {MASK_SET}")

cols = [
    "mouse_id", "experimental_group",
    "refspace_full_roi_px", "refspace_buffered_roi_px", "refspace_technical_valid_px",
    "canonical_full_area_px", "canonical_buffered_area_px", "canonical_technical_valid_px",
    "technical_coverage_fraction",
    "cohort_reference_technical_coverage",
    "technical_relative_to_cohort",
    "technical_relative_to_cohort_low",
    "technical_relative_to_cohort_high",
    "technical_pixel_pass",
    "technical_pass",
]
display(roi[[c for c in cols if c in roi.columns]].sort_values("mouse_id").reset_index(drop=True))

thr = s6b.thresholds_for_mask_set(config, MASK_SET)
rel_cfg = thr["cohort_relative"]

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(roi))
vals = roi["technical_coverage_fraction"].to_numpy(float)
ax.scatter(x, vals)
ax.set_xticks(x)
ax.set_xticklabels(roi["mouse_id"], rotation=45, ha="right")

cohort_med = float(roi["cohort_reference_technical_coverage"].dropna().iloc[0])
ax.axhline(cohort_med, linestyle="--", label="cohort median")
ax.axhline(
    cohort_med * float(rel_cfg["min_relative_fraction"]),
    linestyle=":",
    label="lower relative gate",
)
ax.axhline(
    cohort_med * float(rel_cfg["max_relative_fraction"]),
    linestyle=":",
    label="upper QC flag",
)
ax.set_ylim(0, 1.02)
ax.set_ylabel("technical coverage fraction")
ax.set_title(f"{MASK_SET} | {QC_ROI}")
ax.legend()
plt.show()


## 7. ROI eligibility for the selected analysis

`include=True` denotes a **cohort-level PASS** based on fixed technical coverage across mice.
Individual `homotopic_stem_pass` and `global_stem_pass` fields apply to specific recording stems and can differ from cohort-level ROI eligibility.
`all_selected_stems_pass` is diagnostic only; it is **not** a cohort-level ROI-selection criterion.


In [ ]:
selected = result.get("selected_rois", pd.DataFrame()).copy()
selected = selected[selected["mask_set"].astype(str).eq(MASK_SET)].copy()

display(
    selected.sort_values(["analysis", "broad_acronym", "hemisphere"])
    .reset_index(drop=True)
)

funnel = result.get("completeness_funnel", pd.DataFrame()).copy()
funnel = funnel[funnel["mask_set"].astype(str).eq(MASK_SET)].copy()

display(
    funnel.sort_values(["analysis", "broad_acronym", "hemisphere"])
    .reset_index(drop=True)
)


## 8. One mouse: the three spatial masks in canonical space

For the selected `QC_MOUSE × MASK_SET`:

- `bilateral` — valid left and right buffered Allen cortex independently;
- `global_healthy` — only the healthy hemisphere;
- `homotopic_pairable_display` — only positions with a valid mirrored partner, stored on the configured display hemisphere.


In [ ]:
mouse_rows = side[side["mouse_id"].astype(str).eq(str(QC_MOUSE))].copy()
if mouse_rows.empty:
    raise ValueError(f"No {MASK_SET} rows for mouse={QC_MOUSE}")

gids = mouse_rows["group_id"].astype(str).unique().tolist()
if len(gids) != 1:
    raise ValueError(f"Expected one group_id, got {gids}")
QC_GROUP_ID = gids[0]

mouse_can_dir = out_paths["canonical_masks_root"] / s6b.safe_key(MASK_SET, QC_GROUP_ID)

files = {
    "fixed technical": mouse_can_dir / "fixed_technical_canonical.npy",
    "bilateral": mouse_can_dir / "bilateral_fixed_technical_canonical.npy",
    "global healthy": mouse_can_dir / "global_healthy_fixed_technical_canonical.npy",
    "homotopic pairable display": mouse_can_dir / "homotopic_pairable_display_canonical.npy",
}

masks = {}
for name, path in files.items():
    if not path.exists():
        raise FileNotFoundError(path)
    masks[name] = np.load(path).astype(bool)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, mask) in zip(axes, masks.items()):
    ax.imshow(np.ma.masked_where(canonical["full_side_labels"] == 0, canonical["full_side_labels"]),
              cmap="gray", alpha=0.22)
    ax.imshow(np.where(mask, 1, np.nan), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"{QC_MOUSE}\n{name}\npx={int(mask.sum())}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Common-brain edge erosion QC

This section documents the optional boundary exclusion introduced in Step6b v4.12 relative to the earlier v4.10 implementation.

For each `mask_set × mouse`, the operation is:

```text
common_brain
→ optional erosion by step6b_fc_geometry.technical_geometry.common_brain_edge_erosion.radius_px
→ subtract fixed technical artifacts
→ fixed_technical_ref
```

`left_hemisphere_ref.npy` and `right_hemisphere_ref.npy` retain their anatomical left/right definition and use `step6b_fc_geometry.hemisphere.midline_exclusion_px` independently of the brain-edge erosion.

Existing authoritative output filenames are retained. The additional `technical_common_brain_eroded_ref.npy` and `technical_common_brain_edge_excluded_ref.npy` files support QC and auditing.


In [ ]:
mask_set_ref_dir = out_paths["mask_sets_root"] / s6b.safe_key(MASK_SET, QC_GROUP_ID)

common_path = mask_set_ref_dir / "technical_common_brain_ref.npy"
eroded_path = mask_set_ref_dir / "technical_common_brain_eroded_ref.npy"
edge_path = mask_set_ref_dir / "technical_common_brain_edge_excluded_ref.npy"
fixed_path = mask_set_ref_dir / "fixed_technical_mask_ref.npy"

for p in [common_path, eroded_path, edge_path, fixed_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing v4.12 common-brain QC output: {p}")

common_ref = np.load(common_path).astype(bool)
eroded_ref = np.load(eroded_path).astype(bool)
edge_excluded_ref = np.load(edge_path).astype(bool)
fixed_ref = np.load(fixed_path).astype(bool)

edge_cfg = s6b.cfg_get(
    config,
    "step6b_fc_geometry",
    "technical_geometry",
    "common_brain_edge_erosion",
    default={},
) or {}

print("common_brain_edge_erosion config:", edge_cfg)
print("common brain px:", int(common_ref.sum()))
print("after erosion px:", int(eroded_ref.sum()))
print("excluded edge px:", int(edge_excluded_ref.sum()))
print("fixed technical px after fixed artifacts:", int(fixed_ref.sum()))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, arr, title in [
    (axes[0], common_ref, "common brain (original)"),
    (axes[1], edge_excluded_ref, "excluded edge band"),
    (axes[2], eroded_ref, "common brain after erosion"),
    (axes[3], fixed_ref, "fixed technical after artifacts"),
]:
    ax.imshow(arr.astype(float), interpolation="nearest")
    ax.set_title(f"{title}\npx={int(arr.sum())}")
    ax.axis("off")
plt.suptitle(f"{MASK_SET} | {QC_MOUSE} | Step6b v4.12 common-brain edge QC")
plt.tight_layout()
plt.show()


## 9. Reference-space analytical atlas repair — authoritative Step7 atlas QC

For the selected `QC_MOUSE × MASK_SET`, Step6b saves:

```text
analysis_roi_ref_raw.npy
    -> repair enclosed zero/non-zero holes
analysis_roi_ref_repaired.npy
    -> boundary erosion
analysis_roi_ref_buffered_repaired.npy
```

`analysis_roi_ref_buffered_repaired.npy` is the **authoritative reference-space analytical atlas for Step7**.

The legacy `refspace_broad_buffered_qc.npy` is retained for compatibility and QC. It should not be used as the Step7 analytical atlas source.


In [ ]:
mask_set_ref_dir = out_paths["mask_sets_root"] / s6b.safe_key(MASK_SET, QC_GROUP_ID)

raw_ref_path = mask_set_ref_dir / "analysis_roi_ref_raw.npy"
repaired_ref_path = mask_set_ref_dir / "analysis_roi_ref_repaired.npy"
buffered_ref_path = mask_set_ref_dir / "analysis_roi_ref_buffered_repaired.npy"
legacy_ref_path = mask_set_ref_dir / "refspace_broad_buffered_qc.npy"
local_repair_csv = mask_set_ref_dir / "reference_space_hole_repairs.csv"

for p in [raw_ref_path, repaired_ref_path, buffered_ref_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing v4.12 reference atlas output: {p}\n"
            "Run Step6b v4.12 with RUN_BATCH=True first."
        )

raw_ref = np.load(raw_ref_path).astype(np.int32)
repaired_ref = np.load(repaired_ref_path).astype(np.int32)
buffered_ref = np.load(buffered_ref_path).astype(np.int32)
legacy_ref = np.load(legacy_ref_path).astype(np.int32) if legacy_ref_path.exists() else None
changed_ref = raw_ref != repaired_ref

print("Authoritative Step7 atlas:", buffered_ref_path)
print("raw shape:", raw_ref.shape)
print("n repaired reference pixels:", int(changed_ref.sum()))

if local_repair_csv.exists():
    ref_repairs = pd.read_csv(local_repair_csv)
    print("Reference-space repair audit:")
    display(ref_repairs.reset_index(drop=True))
else:
    ref_repairs = pd.DataFrame()
    print("No local reference-space repair CSV (no candidates or old run).")

# Full-frame comparison.
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, arr, title in [
    (axes[0], raw_ref, "RAW analytical atlas"),
    (axes[1], repaired_ref, "REPAIRED analytical atlas"),
    (axes[2], buffered_ref, "BUFFERED after repair\nAUTHORITATIVE FOR STEP7"),
    (axes[3], legacy_ref if legacy_ref is not None else np.zeros_like(raw_ref), "legacy raw->buffer QC"),
]:
    ax.imshow(s6b.compact_label_display(arr), interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")

if changed_ref.any():
    axes[1].imshow(np.where(changed_ref, 1.0, np.nan), cmap="Reds", alpha=0.9, interpolation="nearest")

plt.suptitle(f"{MASK_SET} | {QC_MOUSE} | reference analytical atlas")
plt.tight_layout()
plt.show()

# Zoom automatically around actual repaired pixels; a one-pixel MOs repair is visible here.
if changed_ref.any():
    yy, xx = np.where(changed_ref)
    pad = 25
    y0 = max(0, int(yy.min()) - pad)
    y1 = min(raw_ref.shape[0], int(yy.max()) + pad + 1)
    x0 = max(0, int(xx.min()) - pad)
    x1 = min(raw_ref.shape[1], int(xx.max()) + pad + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(s6b.compact_label_display(raw_ref[y0:y1, x0:x1]), interpolation="nearest")
    axes[0].set_title("RAW zoom")

    axes[1].imshow(s6b.compact_label_display(repaired_ref[y0:y1, x0:x1]), interpolation="nearest")
    axes[1].imshow(
        np.where(changed_ref[y0:y1, x0:x1], 1.0, np.nan),
        cmap="Reds", alpha=0.95, interpolation="nearest",
    )
    axes[1].set_title("REPAIRED zoom\nred = changed pixel(s)")

    axes[2].imshow(s6b.compact_label_display(buffered_ref[y0:y1, x0:x1]), interpolation="nearest")
    axes[2].set_title("BUFFERED REPAIRED zoom")

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No reference analytical-atlas repairs for this mouse/mask set.")


## 10. One stem: shadow → pathology → final usable in canonical space

In [ ]:
stem_table = result.get("side_stem_usable_coverage", pd.DataFrame()).copy()
stem_table = stem_table[
    stem_table["mask_set"].astype(str).eq(MASK_SET)
    & stem_table["mouse_id"].astype(str).eq(str(QC_MOUSE))
].copy()

available_stems = stem_table[["slot", "stem", "exp_day"]].drop_duplicates().reset_index(drop=True)
display(available_stems)

if QC_STEM is None:
    if available_stems.empty:
        raise ValueError("No stems available")
    STEM = str(available_stems.iloc[0]["stem"])
else:
    STEM = str(QC_STEM)

print("Showing stem:", STEM)

stem_can_dir = mouse_can_dir / "by_stem" / s6b.safe_key(STEM)

stem_files = {
    "fixed technical": mouse_can_dir / "bilateral_fixed_technical_canonical.npy",
    "rescue shadow": stem_can_dir / "rescue_shadow_canonical.npy",
    "pathology": stem_can_dir / "pathology_canonical.npy",
    "final usable": stem_can_dir / "bilateral_final_usable_canonical.npy",
}

stem_masks = {name: np.load(path).astype(bool) for name, path in stem_files.items()}

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, mask) in zip(axes, stem_masks.items()):
    ax.imshow(np.ma.masked_where(canonical["full_side_labels"] == 0, canonical["full_side_labels"]),
              cmap="gray", alpha=0.20)
    ax.imshow(np.where(mask, 1, np.nan), cmap="viridis", vmin=0, vmax=1)
    ax.set_title(f"{name}\npx={int(mask.sum())}")
    ax.axis("off")
plt.suptitle(f"{QC_MOUSE} | {MASK_SET} | {STEM}")
plt.tight_layout()
plt.show()


## 11. Selected ROI across stems

This is the key check for the corrected rule: one failed stem is shown as `side_stem_pass=False`, while other stems remain usable.


In [ ]:
roi_stems = stem_table[stem_table["roi_key"].astype(str).eq(QC_ROI)].copy()

cols = [
    "slot", "stem", "exp_day",
    "canonical_buffered_area_px",
    "canonical_fixed_technical_px",
    "canonical_fixed_technical_coverage_fraction",
    "canonical_shadow_lost_px",
    "canonical_pathology_lost_px",
    "canonical_final_usable_px",
    "stem_usable_coverage_fraction",
    "final_usable_fraction_of_fixed_technical",
    "technical_pass",
    "side_stem_pass",
]
display(
    roi_stems[[c for c in cols if c in roi_stems.columns]]
    .sort_values(["slot", "stem"])
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(9, 4))
plot = roi_stems.reset_index(drop=True)
x = np.arange(len(plot))
ax.bar(x, plot["stem_usable_coverage_fraction"].to_numpy(float))
ax.set_xticks(x)
ax.set_xticklabels(plot["exp_day"].astype(str), rotation=45, ha="right")

stem_thr = s6b.thresholds_for_mask_set(config, MASK_SET)["min_stem_usable_coverage"]
ax.axhline(float(stem_thr), linestyle="--", label="stem usable threshold")
ax.set_ylim(0, 1.02)
ax.set_ylabel("usable fraction of full buffered canonical ROI")
ax.set_title(f"{QC_MOUSE} | {QC_ROI}: stem-specific usable coverage")
ax.legend()
plt.show()


## Main outputs to carry forward

For Step7 / statistics, the key tables are:

- `roi_side_technical_coverage_by_mouse.csv` — fixed representation of `ROI_left/right`;
- `roi_homotopic_pair_coverage_by_mouse.csv` — fixed mirrored-pair representation;
- `roi_side_stem_usable_coverage.csv` — side ROI availability per stem;
- `roi_homotopic_stem_usable_coverage.csv` — homotopic pair availability per stem;
- `roi_global_stem_usable_coverage.csv` — GBC-side availability per stem;
- `selected_rois_by_mask_set.csv` — ROI-level inclusion based on technical representation across mice;
- `cohort_frequency_roi_summary.csv` and canonical frequency maps — experiment-level spatial QC.


## 12. Compare independent REST branches — mouse-specific fixed masks and cohort coverage

These extra checks do not change saved masks. All original one-mouse canonical/reference visualization cells remain available above.

In [ ]:
# Optional side-by-side fixed technical mask comparison for the SAME mouse.
from IPython.display import display
from pathlib import Path
branch_tables = {v: s6b.load_step6b_tables(CONFIG_PATH, rest_variant=v) for v in ("without_rest", "with_rest")}
for v, tb in branch_tables.items():
    sel = tb.get("selection", pd.DataFrame())
    chosen = sel[(sel.get("mouse_id", pd.Series(dtype=str)).astype(str) == str(QC_MOUSE))
                 & (sel.get("mask_set", pd.Series(dtype=str)).astype(str) == MASK_SET)] if not sel.empty else pd.DataFrame()
    print(v, "OK slots:", chosen.loc[chosen["status"].eq("ok"), "slot"].tolist() if not chosen.empty else [])

files = {}
for v in ("without_rest", "with_rest"):
    config["step6b_fc_geometry"]["active_rest_variant"] = v
    p = s6b.build_step6b_paths(config, paths)["canonical_masks_root"] / s6b.safe_key(MASK_SET, QC_GROUP_ID) / "fixed_technical_canonical.npy"
    if p.exists():
        files[v] = np.load(p).astype(bool)
if files:
    fig, axs = plt.subplots(1, len(files), figsize=(6 * len(files), 5), squeeze=False)
    for ax, (v, m) in zip(axs.ravel(), files.items()):
        ax.imshow(np.where(m, 1, np.nan), cmap="viridis", vmin=0, vmax=1)
        ax.set_title(f"{QC_MOUSE} {MASK_SET} — {v}\nN={int(m.sum())}")
        ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    print("No saved masks yet: set RUN_BATCH=True and execute earlier cells.")
